In [1]:
import torch
import torch.nn as nn

In [2]:
inputs_posttrained = torch.tensor([ [0.44, 0.15, 0.89], #Your
                                    [0.55, 0.87, 0.66], #journey
                                    [0.53, 0.85, 0.67], #starts
                                    [0.22, 0.58, 0.33], #with
                                    [0.77, 0.25, 0.10], #one
                                    [0.05, 0.80, 0.55]])  #step

dim_in = inputs_posttrained.shape[1]

dim_out = 2

A function that sums up self attention

In [3]:
class SelfAttentionV2(nn.Module):

    def __init__ (self, dim_in, dim_out, qkv_bias = False):
    
        super().__init__()
        self.W_query = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.W_key = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias = qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attention_scores = attn_scores/ (keys.shape[-1] ** 0.5)
        attn_weights = torch.softmax(attention_scores, dim = -1)

        context_vect = attn_weights @ values
        
        return context_vect

In [4]:
torch.manual_seed(42)

self_attn_V2 = SelfAttentionV2(dim_in, dim_out)

print(self_attn_V2(inputs_posttrained))


tensor([[0.3748, 0.2789],
        [0.3755, 0.2843],
        [0.3756, 0.2839],
        [0.3761, 0.2774],
        [0.3748, 0.2848],
        [0.3765, 0.2756]], grad_fn=<MmBackward0>)


In [5]:
queries = self_attn_V2.W_query(inputs_posttrained)

keys = self_attn_V2.W_key(inputs_posttrained)

attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim = 1)

print(attn_weights)

tensor([[0.1601, 0.1725, 0.1726, 0.1679, 0.1467, 0.1802],
        [0.1621, 0.1777, 0.1779, 0.1644, 0.1303, 0.1876],
        [0.1622, 0.1773, 0.1776, 0.1645, 0.1314, 0.1870],
        [0.1658, 0.1724, 0.1726, 0.1652, 0.1474, 0.1766],
        [0.1589, 0.1774, 0.1776, 0.1660, 0.1310, 0.1892],
        [0.1679, 0.1714, 0.1715, 0.1646, 0.1509, 0.1736]],
       grad_fn=<SoftmaxBackward0>)


In [6]:
contxt_len = attn_weights.shape[0]

simple_mask = torch.tril(torch.ones(contxt_len, contxt_len))

print(simple_mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [7]:
masked_simple = attn_weights * simple_mask

print(masked_simple)

tensor([[0.1601, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1621, 0.1777, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1622, 0.1773, 0.1776, 0.0000, 0.0000, 0.0000],
        [0.1658, 0.1724, 0.1726, 0.1652, 0.0000, 0.0000],
        [0.1589, 0.1774, 0.1776, 0.1660, 0.1310, 0.0000],
        [0.1679, 0.1714, 0.1715, 0.1646, 0.1509, 0.1736]],
       grad_fn=<MulBackward0>)


In [8]:
row_sums = masked_simple.sum(dim = 1, keepdim=True)

normalized = masked_simple.divide(row_sums)

print(normalized)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4771, 0.5229, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3137, 0.3429, 0.3434, 0.0000, 0.0000, 0.0000],
        [0.2453, 0.2551, 0.2553, 0.2444, 0.0000, 0.0000],
        [0.1960, 0.2188, 0.2190, 0.2048, 0.1615, 0.0000],
        [0.1679, 0.1714, 0.1715, 0.1646, 0.1509, 0.1736]],
       grad_fn=<DivBackward0>)


Dropout: Concept to essentially "turnoff" a select amount of neurons in Deep Learning. We apply the same here by initializing a dropout rate of 50% (meaning we mask 50% of each row) 



Example:

In [9]:
torch.manual_seed(42)

dropout = nn.Dropout(0.5)

example = torch.ones(6,6)

print(dropout(example))

tensor([[2., 2., 2., 2., 0., 2.],
        [0., 0., 2., 2., 2., 2.],
        [0., 0., 2., 0., 2., 0.],
        [0., 2., 2., 0., 2., 2.],
        [2., 2., 0., 2., 2., 2.],
        [2., 2., 2., 2., 0., 0.]])


Ensure that the code can handle batches of more than one input

In [14]:
batch = torch.stack((inputs_posttrained, inputs_posttrained), dim = 0)

print(batch.shape) #two inputs and the 6x3 is the data for the corresponding input

torch.Size([2, 6, 3])


In [18]:
class CausalAttention(nn.Module):

    def __init__ (self, dim_in, dim_out, context_len, dropout, qkv_bias = False):

        super().__init__()
        self.dim_out = dim_out
        self.W_query = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.W_key = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.Dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_len, context_len), diagonal=1))

    def forward(self, x):
        batch_dim, num_tokens, dim_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim = -1)
        attn_weights = self.Dropout(attn_weights)

        context_vect = attn_weights @ values
        return context_vect

In [20]:
torch.manual_seed(42)

context_len = batch.shape[1]
ca = CausalAttention(dim_in, dim_out, context_len, 0.0)
context_vecs = ca(batch)
print(context_vecs)

tensor([[[0.4472, 0.1068],
         [0.4677, 0.2595],
         [0.4716, 0.3049],
         [0.4125, 0.2936],
         [0.4070, 0.2582],
         [0.3765, 0.2756]],

        [[0.4472, 0.1068],
         [0.4677, 0.2595],
         [0.4716, 0.3049],
         [0.4125, 0.2936],
         [0.4070, 0.2582],
         [0.3765, 0.2756]]], grad_fn=<UnsafeViewBackward0>)
